In [1]:
import numpy as np
import matplotlib.pyplot as plt

Lx = 1.0
u_left, u_right = 0.0, 2.0

def f(x):
    return -np.pi**2 * np.sin(np.pi * x)

def u_ref(x):
    return np.sin(np.pi * x) + 2 * x

def solve_poisson(dx):
    """Возвращает координаты центров контрольных объёмов и решение в них."""
    if not np.isfinite(dx) or dx <= 0:
        raise ValueError("Шаг должен быть конечным и положительным.")
    n = max(1, round(Lx / dx))
    h = Lx / n
    x = (np.arange(n) + 0.5) * h

    # Собираем баланс потоков, умноженный на h.
    A = np.zeros((n, n))
    b = f(x) * h**2
    for i in range(n - 1):
        A[i, i] -= 1
        A[i, i + 1] += 1
        A[i + 1, i] += 1
        A[i + 1, i + 1] -= 1

    A[0, 0] -= 2
    A[-1, -1] -= 2
    b[0] -= 2 * u_left
    b[-1] -= 2 * u_right
    return x, np.linalg.solve(A, b)

In [ ]:
x, u = solve_poisson(0.05)
print(f"Максимальная ошибка в центрах ячеек: {np.max(np.abs(u - u_ref(x))):.8f}")
x_ref = np.linspace(0, Lx, 300)
plt.plot(x_ref, u_ref(x_ref), label="Точное решение")
plt.scatter(x, u, label="МКO (центры ячеек)", color="red", s=16)
plt.scatter([0, Lx], [u_left, u_right], marker="s", label="Граничные условия")
plt.xlabel("x")
plt.ylabel("u")
plt.legend()
plt.grid()
plt.show()

In [ ]:
steps = 0.1 / 2**np.arange(8)
errors = []
print("    h          Макс. ошибка     Порядок")
for h in steps:
    x, u = solve_poisson(h)
    error = np.max(np.abs(u - u_ref(x)))
    order = f"{np.log2(errors[-1] / error):.3f}" if errors else "—"
    print(f"{h:10.8f}     {error:.4e}       {order}")
    errors.append(error)

plt.loglog(steps, errors, "ro-", label="Ошибка МКО")
plt.loglog(steps, errors[0] * (steps / steps[0])**2, "k--", label=r"$C h^2$")
plt.xlabel("Шаг сетки h")
plt.ylabel("Максимальная ошибка в центрах ячеек")
plt.grid(True, which="both")
plt.legend()
plt.show()